In [1]:
from pathlib import Path

print("Notebook working folder:", Path.cwd())

for file in sorted(Path.cwd().glob("*.parquet")):
    print(file.name)

Notebook working folder: /Users/babiakattack/anaconda_projects/5ca9e9e1-5ea4-49bc-9947-a3ec2c301fc2


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path(".")

# Check that all seven files are available
required_files = [
    "X_train_processed.parquet",
    "X_val_processed.parquet",
    "X_test_processed.parquet",
    "y_train.parquet",
    "y_val.parquet",
    "y_test.parquet",
    "ontario_macro_series.parquet",
]

missing_files = [
    name for name in required_files
    if not (DATA_DIR / name).is_file()
]

if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")

# Load predictors and corresponding default outcomes
X_train_sc = pd.read_parquet(DATA_DIR / "X_train_processed.parquet")
X_val_sc = pd.read_parquet(DATA_DIR / "X_val_processed.parquet")
X_test_sc = pd.read_parquet(DATA_DIR / "X_test_processed.parquet")

y_train = pd.read_parquet(DATA_DIR / "y_train.parquet")["loan_status"]
y_val = pd.read_parquet(DATA_DIR / "y_val.parquet")["loan_status"]
y_test = pd.read_parquet(DATA_DIR / "y_test.parquet")["loan_status"]

df_macro = pd.read_parquet(DATA_DIR / "ontario_macro_series.parquet")

# Verify the modelling data
summary = []

for name, X_split, y_split in [
    ("Training", X_train_sc, y_train),
    ("Validation", X_val_sc, y_val),
    ("Test", X_test_sc, y_test),
]:
    assert X_split.index.equals(y_split.index), f"{name}: indexes do not match"
    assert X_split.isna().sum().sum() == 0, f"{name}: missing predictors"
    assert y_split.isna().sum() == 0, f"{name}: missing outcomes"
    assert np.isfinite(X_split.to_numpy(dtype=float)).all(), (
        f"{name}: infinite or invalid predictors"
    )
    assert y_split.isin([0, 1]).all(), f"{name}: outcomes must be 0 or 1"

    summary.append({
        "Sample": name,
        "Borrowers": len(y_split),
        "Predictors": X_split.shape[1],
        "Default rate": f"{y_split.mean():.2%}",
    })

assert X_train_sc.columns.equals(X_val_sc.columns), "Validation columns differ"
assert X_train_sc.columns.equals(X_test_sc.columns), "Test columns differ"

assert df_macro.isna().sum().sum() == 0, "Missing economic observations"
assert {
    "Date",
    "Ontario_Unemployment_Rate",
    "Ontario_CPI_YoY",
}.issubset(df_macro.columns), "Required economic columns are missing"

display(pd.DataFrame(summary))
print(f"Ontario economic data: {len(df_macro)} observations")
print("\nAll seven files loaded and passed the checks.")

,Sample,Borrowers,Predictors,Default rate
0,Training,22806,20,21.81%
1,Validation,4887,20,21.81%
2,Test,4888,20,21.83%


Ontario economic data: 107 observations

All seven files loaded and passed the checks.


In [5]:
from sklearn.linear_model import LogisticRegression

# Set up logistic regression with L2 regularization
logit_model = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=5000,
    random_state=42,
)

# Learn the coefficients from the training data
logit_model.fit(X_train_sc, y_train)

# Predict default probabilities for the validation sample
logit_val_prob = logit_model.predict_proba(X_val_sc)[:, 1]

# Classify borrowers using a 50% probability threshold
logit_val_pred = (logit_val_prob >= 0.50).astype(int)

print("Logistic regression fitted successfully.")
print(f"Number of predictors: {logit_model.n_features_in_}")
print(f"Optimization iterations: {logit_model.n_iter_[0]}")
print(
    "Average predicted validation default probability: "
    f"{logit_val_prob.mean():.2%}"
)
print(f"Observed validation default rate: {y_val.mean():.2%}")

Logistic regression fitted successfully.
Number of predictors: 20
Optimization iterations: 55
Average predicted validation default probability: 22.21%
Observed validation default rate: 21.81%


In [6]:
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Baseline probability: training sample's observed default rate
baseline_val_prob = np.full(len(y_val), y_train.mean())

# Evaluate validation probability predictions
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Constant Probability"],
    "Brier Score": [
        brier_score_loss(y_val, logit_val_prob),
        brier_score_loss(y_val, baseline_val_prob),
    ],
    "Log Loss": [
        log_loss(y_val, logit_val_prob),
        log_loss(y_val, baseline_val_prob),
    ],
    "ROC-AUC": [
        roc_auc_score(y_val, logit_val_prob),
        roc_auc_score(y_val, baseline_val_prob),
    ],
})

display(results.set_index("Model").round(4))

# Evaluate classifications at the 50% threshold
print(f"Accuracy:  {accuracy_score(y_val, logit_val_pred):.2%}")
print(
    f"Precision: "
    f"{precision_score(y_val, logit_val_pred, zero_division=0):.2%}"
)
print(
    f"Recall:    "
    f"{recall_score(y_val, logit_val_pred, zero_division=0):.2%}"
)
print(f"F1 score:  {f1_score(y_val, logit_val_pred, zero_division=0):.4f}")

cm = confusion_matrix(y_val, logit_val_pred, labels=[0, 1])

display(pd.DataFrame(
    cm,
    index=["Actual non-default", "Actual default"],
    columns=["Predicted non-default", "Predicted default"],
))

,Brier Score,Log Loss,ROC-AUC
Model,,,
Logistic Regression,0.1022,0.3409,0.8705
Constant Probability,0.1705,0.5245,0.5000


Accuracy:  86.94%
Precision: 77.58%
Recall:    56.47%
F1 score:  0.6536


,Predicted non-default,Predicted default
Actual non-default,3647,174
Actual default,464,602
